<a href="https://www.kaggle.com/code/buianhtruc/omnivoice-on-kaggle?scriptVersionId=339822089" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

remember: GPU T4 x2 and on Internet in Session options

Create CLOUDFLARE_TUNNEL_TOKEN or (NGROK_AUTHTOKEN + NGROK_DOMAIN) if you want to use a fixed domain (Add-ons -> Secrets -> Add a new secret)

docs: https://github.com/k2-fsa/OmniVoice

## Optional: HuggingFace token

If a model is gated, create a token at https://huggingface.co/settings/tokens and add to Kaggle Secrets:
  - Label: `HF_TOKEN`    Value: your HuggingFace token (e.g. `hf_...`)

# Setup

In [1]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("  -> HF_TOKEN loaded")
except Exception:
    print("  -> No HF_TOKEN secret (fine for public models)")

print("Tunnel mode:")
print("  1) Quick Tunnel (Cloudflare - khong can cau hinh)")
print("  2) Named Tunnel (can CLOUDFLARE_TUNNEL_TOKEN)")
print("  3) Ngrok (can NGROK_AUTHTOKEN + NGROK_DOMAIN)")
choice = input("Chon (1/2/3): ").strip()
TUNNEL_MODE = {"1": "quick", "2": "named", "3": "ngrok"}.get(choice, "quick")
print(f"Tunnel mode: {TUNNEL_MODE}")

OMNIVOICE_DIR = "/kaggle/working/OmniVoice"

  -> HF_TOKEN loaded
Tunnel mode:
  1) Quick Tunnel (Cloudflare - khong can cau hinh)
  2) Named Tunnel (can CLOUDFLARE_TUNNEL_TOKEN)
  3) Ngrok (can NGROK_AUTHTOKEN + NGROK_DOMAIN)


Chon (1/2/3):  2


Tunnel mode: named


In [2]:
!git clone "https://github.com/k2-fsa/OmniVoice.git"
%cd OmniVoice
!uv sync

Cloning into 'OmniVoice'...
remote: Enumerating objects: 528, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 528 (delta 153), reused 135 (delta 135), pack-reused 306 (from 1)
Receiving objects: 100% (528/528), 1.34 MiB | 31.97 MiB/s, done.
Resolving deltas: 100% (286/286), done.
/kaggle/working/OmniVoice
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 148 packages in 3.62s                                       
Prepared 101 packages in 1m 02s                                          
░░░░░░░░░░░░░░░░░░░░ [0/101] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to sup

In [3]:
import os, stat, urllib.request

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_PATH)
    os.chmod(CLOUDFLARED_PATH, os.stat(CLOUDFLARED_PATH).st_mode | stat.S_IEXEC)

if TUNNEL_MODE == "ngrok":
    import zipfile
    NGROK_PATH = "/kaggle/working/ngrok"
    if not os.path.exists(NGROK_PATH):
        print("Downloading ngrok...")
        urllib.request.urlretrieve(
            "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip",
            NGROK_PATH + ".zip")
        with zipfile.ZipFile(NGROK_PATH + ".zip", "r") as z:
            z.extractall("/kaggle/working")
        os.chmod(NGROK_PATH, os.stat(NGROK_PATH).st_mode | stat.S_IEXEC)

# Start OmniVoice Demo

In [ ]:
import os, subprocess, time, urllib.request, sys

# ================= START DEMO (background) =================

proc = subprocess.Popen(
    ["uv", "run", "omnivoice-demo"],
    cwd=OMNIVOICE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,bufsize=1, env=os.environ.copy(),
)
import threading
def drain():
    for line in proc.stdout:
        print(line, end="", flush=True)   
threading.Thread(target=drain, daemon=True).start()

print("Demo started (PID:", proc.pid, "). Waiting for Gradio...")

# ================= TUNNEL =================
t = None
if TUNNEL_MODE == "quick":
    print("Starting Cloudflare Quick Tunnel...")
    t = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:7860"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
elif TUNNEL_MODE == "named":
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("CLOUDFLARE_TUNNEL_TOKEN")
    if token:
        print("Starting Cloudflare Named Tunnel...")
        t = subprocess.Popen(["cloudflared", "tunnel", "run", "--token", token],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    else:
        print("Missing CLOUDFLARE_TUNNEL_TOKEN secret.")
elif TUNNEL_MODE == "ngrok":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    auth = secrets.get_secret("NGROK_AUTHTOKEN")
    domain = secrets.get_secret("NGROK_DOMAIN")
    if auth:
        subprocess.run(["/kaggle/working/ngrok", "config", "add-authtoken", auth], check=False)
        print(f"Starting ngrok... https://{domain}")
        t = subprocess.Popen(["/kaggle/working/ngrok", "http", f"--domain={domain}", "7860"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    else:
        print("Missing NGROK_AUTHTOKEN secret.")

if t:
    for line in t.stdout:
        print(line, end="")
else:
    print("No tunnel. Local only: http://localhost:7860" )

Demo started (PID: 202 ). Waiting for Gradio...
Starting Cloudflare Named Tunnel...
b'Error in sitecustomize; set PYTHONVERBOSE for traceback:\n'b"ModuleNotFoundError: No module named 'wrapt'\n"2026-08-03T04:45:53Z INF Starting tunnel tunnelID=ec67074d-e4f9-40e7-ab45-bf553ff42844
2026-08-03T04:45:53Z INF Version 2026.7.3 (Checksum 9d71c677db00134c1bd4144b7783486b654ad281b1ea62b4972098d19f770f17)
2026-08-03T04:45:53Z INF GOOS: linux, GOVersion: go1.26.4, GoArch: amd64
2026-08-03T04:45:53Z INF Settings: map[token:*****]
2026-08-03T04:45:53Z INF Autoupdate frequency is set autoupdateFreq=86400000
2026-08-03T04:45:53Z INF Generated Connector ID: 635ff5e6-fa92-4ad1-b731-13580e730d0d
2026-08-03T04:45:53Z INF Initial protocol quic
2026-08-03T04:45:53Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-08-03T04:45:53Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026/08/03 04:45:53 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 41